# Convert `graph.mat` to DGL `graph.bin`

This notebook converts the old MCCE-MHGCN `.mat` graph format into the DGL `.bin` heterograph format used by the current training script.

It writes:

- node features to `g.nodes[ntype].data["feat"]`
- target edge split masks to `g.edges[target_etype].data["train_mask"]`, `val_mask`, and `test_mask`
- a DGL heterograph saved by `dgl.save_graphs(...)`


## 1. Environment

The model repository expects PyTorch 2.3.0 and DGL >= 2.1.0. Install the DGL build that matches your CUDA environment.

In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
from scipy import sparse
from scipy.io import loadmat

try:
    import dgl
except ImportError as exc:
    raise ImportError('Install DGL >= 2.1.0 before running this notebook.') from exc

print('torch:', torch.__version__)
print('dgl:', dgl.__version__)

## 2. Configure paths and schema

Set `GRAPH_MAT` to your old `.mat` file. If you have split files, set `TRAIN_TXT`, `VALID_TXT`, and `TEST_TXT`.

For AMiner-style data the default layer order is usually `paper`, `author`, `affiliation`. You may change `LAYER_NAMES` to match your graph.

In [ ]:
GRAPH_MAT = Path('data/graph.mat')
OUTPUT_BIN = Path('data/graph.bin')
OUTPUT_META = Path('data/graph_bin_metadata.json')

# Optional old split files. Leave as None if graph.mat already contains edge_index/edge_label,
# or if you want to add masks manually later.
TRAIN_TXT = Path('data/train.txt')
VALID_TXT = Path('data/valid.txt')
TEST_TXT = Path('data/test.txt')

# Set paths to None when unavailable.
TRAIN_TXT = TRAIN_TXT if TRAIN_TXT.exists() else None
VALID_TXT = VALID_TXT if VALID_TXT.exists() else None
TEST_TXT = TEST_TXT if TEST_TXT.exists() else None

# Fallback names used only if graph.mat does not contain layer_names.
LAYER_NAMES = ['paper', 'author', 'affiliation']

# Target relation used for link prediction masks.
# Example author cooperation within layer 1: source_layer=1, target_layer=1.
SOURCE_LAYER = 1
TARGET_LAYER = 1
TARGET_RELATION_NAME = 'coauthor'

# If True, intra-layer target split edges are treated as undirected and reverse edges
# receive the same split mask when present in the graph.
UNDIRECTED_TARGET = True

print('graph mat:', GRAPH_MAT.resolve())
print('output bin:', OUTPUT_BIN.resolve())
print('train/valid/test:', TRAIN_TXT, VALID_TXT, TEST_TXT)

## 3. Load MATLAB cell arrays

In [ ]:
def unwrap_cell_value(value):
    while isinstance(value, np.ndarray) and value.dtype == object:
        if value.size == 0:
            return value
        value = value.ravel()[0]
    return value


def cell_to_list(value):
    if isinstance(value, np.ndarray) and value.dtype == object:
        return [unwrap_cell_value(value.ravel()[i]) for i in range(value.size)]
    if isinstance(value, (list, tuple)):
        return [unwrap_cell_value(item) for item in value]
    return [unwrap_cell_value(value)]


def empty_to_none(value):
    if value is None:
        return None
    while isinstance(value, np.ndarray) and value.dtype == object:
        if value.size == 0:
            return None
        value = value.ravel()[0]
    if isinstance(value, np.ndarray):
        if value.size == 0:
            return None
        if value.ndim == 0:
            value = value.item()
        elif not sparse.issparse(value):
            return sparse.csr_matrix(value)
    return value


def cell_to_matrix_grid(value, num_layers):
    if not (isinstance(value, np.ndarray) and value.dtype == object):
        raise ValueError('cross_adj must be a MATLAB cell/object array')
    grid = [[None for _ in range(num_layers)] for _ in range(num_layers)]
    if value.shape[0] == num_layers and value.shape[1] == num_layers:
        for target in range(num_layers):
            for source in range(num_layers):
                grid[target][source] = empty_to_none(value[target, source])
        return grid
    flat = value.ravel()
    if len(flat) != num_layers * num_layers:
        raise ValueError('cross_adj size must be num_layers x num_layers')
    for target in range(num_layers):
        for source in range(num_layers):
            grid[target][source] = empty_to_none(flat[target * num_layers + source])
    return grid


def parse_layer_names(data, num_layers, fallback):
    if 'layer_names' not in data:
        return fallback[:num_layers] if len(fallback) >= num_layers else [f'layer_{i}' for i in range(num_layers)]
    raw = data['layer_names'].ravel()
    names = []
    for value in raw:
        if isinstance(value, np.ndarray):
            value = value.ravel()[0] if value.size else ''
        names.append(str(value))
    return names if len(names) == num_layers else fallback[:num_layers]


## 4. Build a DGL heterograph

In [ ]:
def as_csr(matrix):
    if sparse.issparse(matrix):
        return matrix.tocsr().astype(np.float32)
    return sparse.csr_matrix(matrix, dtype=np.float32)


def matrix_to_edges(matrix):
    matrix = as_csr(matrix).tocoo()
    src = torch.from_numpy(matrix.col.astype(np.int64))
    dst = torch.from_numpy(matrix.row.astype(np.int64))
    return src, dst


def relation_name(source_name, target_name, source_idx, target_idx, is_intra=False):
    if is_intra and source_idx == SOURCE_LAYER and target_idx == TARGET_LAYER:
        return TARGET_RELATION_NAME
    if is_intra:
        return f'{source_name}_to_{target_name}'
    return f'{source_name}_to_{target_name}'


mat = loadmat(GRAPH_MAT)
features_by_layer = cell_to_list(mat['features_by_layer']) if 'features_by_layer' in mat else cell_to_list(mat['features'])
intra_adj = [as_csr(item) for item in cell_to_list(mat['intra_adj'])]
num_layers = len(features_by_layer)
cross_adj = cell_to_matrix_grid(mat['cross_adj'], num_layers) if 'cross_adj' in mat else [[None for _ in range(num_layers)] for _ in range(num_layers)]
layer_names = parse_layer_names(mat, num_layers, LAYER_NAMES)

if len(set(layer_names)) != len(layer_names):
    raise ValueError(f'Layer names must be unique, got {layer_names}')

data_dict = {}
num_nodes_dict = {}
layer_to_ntype = {idx: name for idx, name in enumerate(layer_names)}

for idx, feature in enumerate(features_by_layer):
    ntype = layer_to_ntype[idx]
    feature_matrix = feature.toarray() if sparse.issparse(feature) else np.asarray(feature)
    num_nodes_dict[ntype] = int(feature_matrix.shape[0])

for layer, matrix in enumerate(intra_adj):
    if matrix.nnz == 0:
        continue
    ntype = layer_to_ntype[layer]
    rel = relation_name(ntype, ntype, layer, layer, is_intra=True)
    data_dict[(ntype, rel, ntype)] = matrix_to_edges(matrix)

for target in range(num_layers):
    for source in range(num_layers):
        if source == target:
            continue
        matrix = cross_adj[target][source]
        if matrix is None:
            continue
        matrix = as_csr(matrix)
        if matrix.nnz == 0:
            continue
        source_name = layer_to_ntype[source]
        target_name = layer_to_ntype[target]
        rel = relation_name(source_name, target_name, source, target, is_intra=False)
        data_dict[(source_name, rel, target_name)] = matrix_to_edges(matrix)

g = dgl.heterograph(data_dict, num_nodes_dict=num_nodes_dict)
print(g)
print('canonical etypes:')
for etype in g.canonical_etypes:
    print(' ', etype, g.num_edges(etype))

## 5. Attach node features

In [ ]:
for idx, feature in enumerate(features_by_layer):
    ntype = layer_to_ntype[idx]
    feature_matrix = feature.toarray() if sparse.issparse(feature) else np.asarray(feature)
    g.nodes[ntype].data['feat'] = torch.as_tensor(feature_matrix, dtype=torch.float32)
    print(ntype, g.nodes[ntype].data['feat'].shape)

## 6. Attach train/validation/test masks to the target edge type

Split text files may contain either `src dst`, `src dst label`, or `type src dst label`. Only positive edges (`label == 1`) are used to create masks.

In [ ]:
target_src_type = layer_to_ntype[SOURCE_LAYER]
target_dst_type = layer_to_ntype[TARGET_LAYER]
target_etype = (target_src_type, TARGET_RELATION_NAME, target_dst_type)
if target_etype not in g.canonical_etypes:
    raise ValueError(f'Target etype {target_etype} not found. Available: {g.canonical_etypes}')

def load_positive_pairs(path):
    pairs = []
    if path is None:
        return pairs
    with open(path, 'r', encoding='utf-8') as file:
        for line in file:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.replace(',', ' ').split()
            values = [int(float(part)) for part in parts]
            if len(values) == 2:
                src, dst = values
                label = 1
            elif len(values) == 3:
                src, dst, label = values
            elif len(values) >= 4:
                _edge_type, src, dst, label = values[:4]
            else:
                raise ValueError(f'Bad edge line: {line}')
            if label == 1:
                pairs.append((src, dst))
    return pairs

src_all, dst_all = g.edges(etype=target_etype)
edge_to_eids = {}
for eid, (src, dst) in enumerate(zip(src_all.tolist(), dst_all.tolist())):
    edge_to_eids.setdefault((src, dst), []).append(eid)

def make_mask(pairs, name):
    mask = torch.zeros(g.num_edges(target_etype), dtype=torch.bool)
    missing = []
    for src, dst in pairs:
        keys = [(src, dst)]
        if UNDIRECTED_TARGET and target_src_type == target_dst_type:
            keys.append((dst, src))
        found = False
        for key in keys:
            for eid in edge_to_eids.get(key, []):
                mask[eid] = True
                found = True
        if not found:
            missing.append((src, dst))
    print(f'{name}: {int(mask.sum())} masked edges; {len(missing)} positive pairs missing from graph')
    if missing[:5]:
        print('  first missing pairs:', missing[:5])
    return mask

train_pairs = load_positive_pairs(TRAIN_TXT)
valid_pairs = load_positive_pairs(VALID_TXT)
test_pairs = load_positive_pairs(TEST_TXT)

if not train_pairs or not valid_pairs or not test_pairs:
    raise ValueError('TRAIN_TXT, VALID_TXT and TEST_TXT must exist and contain positive edges to create masks.')

g.edges[target_etype].data['train_mask'] = make_mask(train_pairs, 'train')
g.edges[target_etype].data['val_mask'] = make_mask(valid_pairs, 'valid')
g.edges[target_etype].data['test_mask'] = make_mask(test_pairs, 'test')

print('target etype:', target_etype)

## 7. Save `graph.bin`

In [ ]:
OUTPUT_BIN.parent.mkdir(parents=True, exist_ok=True)
metadata = {
    'layer_names': layer_names,
    'target_etype': ':'.join(target_etype),
    'source_layer': SOURCE_LAYER,
    'target_layer': TARGET_LAYER,
}
dgl.save_graphs(str(OUTPUT_BIN), [g], labels={'metadata_json': torch.tensor(list(json.dumps(metadata).encode('utf-8')), dtype=torch.uint8)})
OUTPUT_META.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print('saved:', OUTPUT_BIN.resolve())
print('metadata:', OUTPUT_META.resolve())

## 8. Quick reload check

In [ ]:
loaded_graphs, loaded_labels = dgl.load_graphs(str(OUTPUT_BIN))
loaded = loaded_graphs[0]
print(loaded)
for ntype in loaded.ntypes:
    print(ntype, loaded.nodes[ntype].data['feat'].shape)
for key in ['train_mask', 'val_mask', 'test_mask']:
    print(key, int(loaded.edges[target_etype].data[key].sum()))